# SENTINEL AI — Phase 1: model training

**Phase 1: preparation, classifier comparison, separate anomaly detection, final test and artifact export (Steps 1-11).** Execute the final step and preserve the bundle before confirming Phase 1 completion.

Goal: inspect CIC-IDS2017 CSV columns and labels before choosing preprocessing. Run code cells in order. Use a Python 3 CPU runtime; no GPU is required for this step. See the project README for dataset download instructions and the overall plan.

## 1. First executable step

Run this cell to check the runtime. The printed version and directory come from your actual session.

In [ ]:
import sys
from pathlib import Path

print("SENTINEL AI — Phase 1")
print("Python:", sys.version.split()[0])
print("Working directory:", Path.cwd())
print("Setup cell completed. No model has been trained.")

## 2. Install inspection dependencies

Only pandas and NumPy are needed now. These version ranges are starter constraints, not a validated training environment lock. Record the installed versions below. We will pin the exact training and inference environment when models are introduced. If Colab requests a session restart, restart and run from the first cell.

In [ ]:
%pip install -q "pandas>=2.2,<3" "numpy>=1.26,<3"

In [ ]:
import json
import csv
import numpy as np
import pandas as pd

print("pandas:", pd.__version__)
print("NumPy:", np.__version__)

## 3. Use your directly uploaded CSV

If you already uploaded the CSV with Colab's left **Files** panel, it is normally under `/content`. You do not need to mount Google Drive. The next cell selects that folder without uploading anything again.

If you have not uploaded it yet, click the upload icon in the Files panel and wait for it to finish. Upload the extracted `.csv`, not a ZIP. Direct uploads disappear when the runtime resets, so keep your original file on your computer.

For the smaller project, start with one attack-day CSV and inspect its actual labels. After review, we will prepare a roughly **20,000-row** subset preserving class proportions. Do not use the first rows as a training subset.

If the CSV is in Drive instead, see the README's optional Drive-mount cell.

In [ ]:
from pathlib import Path

DATA_DIR = Path("/content")
print("Data directory:", DATA_DIR)

## 4. Find your uploaded CSV

This lists CSV files directly inside `DATA_DIR`, avoiding Colab's unrelated `sample_data` folder. If your file is inside another folder, set `DATA_DIR` to that folder first.

In [ ]:
if not DATA_DIR.is_dir():
    raise FileNotFoundError(f"Data directory does not exist: {DATA_DIR}. Check DATA_DIR.")

csv_paths = sorted(
    path for path in DATA_DIR.iterdir()
    if path.is_file() and path.suffix.lower() == ".csv"
)
if not csv_paths:
    raise FileNotFoundError(
        f"No CSV files found directly in {DATA_DIR}. "
        "Upload the extracted CSV in Colab's Files panel, wait for completion, "
        "or set DATA_DIR to its folder."
    )

for index, path in enumerate(csv_paths):
    print(f"[{index}] {path} ({path.stat().st_size / 1024**2:.1f} MiB)")

## 5. Inspect one CSV

Edit `CSV_PATH` to your chosen file from the list. Initially it selects the first file only for convenience. **This is a preview of at most the first 10,000 rows**, not a random sample, full-dataset analysis, or training input. Ordered records can mean the preview contains only one class.

We strip surrounding whitespace from column names and label values, check that a `Label` column actually exists, and report missing and infinite values without modifying the original file. We do not drop duplicates, choose features, or fit any preprocessing yet.

Repeated feature names receive unique suffixes, with a position-based rename log in the report. All values are retained. Renaming does not prove the columns contain identical values; review them before selecting training features. Ambiguous repeated Label columns still stop inspection.

In [ ]:
import csv
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

DATA_DIR = Path("/content")
csv_files = sorted(
    p for p in DATA_DIR.iterdir()
    if p.is_file() and p.suffix.lower() == ".csv"
)
if not csv_files:
    raise FileNotFoundError("Upload your CSV using Colab's Files panel first.")
if len(csv_files) > 1:
    raise ValueError(
        "Multiple CSV files found. Set CSV_PATH to your dataset's exact path "
        "and replace this file-selection block. Files: " + str([p.name for p in csv_files])
    )
CSV_PATH = csv_files[0]
print("Inspecting:", CSV_PATH)
PREVIEW_ROWS = 10_000

# Inspect the raw header before pandas can rename duplicate columns.
with CSV_PATH.open("r", encoding="utf-8-sig", newline="") as handle:
    header = next(csv.reader(handle), [])
normalized_header = [name.strip() for name in header]
if not normalized_header or any(not name for name in normalized_header):
    raise ValueError("CSV header is empty or contains an unnamed column. Share the header for review.")
# Keep every column, assigning unique names to repeated feature headers.
if sum(name.casefold() == "label" for name in normalized_header) != 1:
    raise ValueError("Expected exactly one Label column in the raw header. Share the header for review.")

reserved_names = set(normalized_header)
used_names = set()
unique_header = []
renamed_columns = []
for position, name in enumerate(normalized_header):
    unique_name = name
    suffix = 2
    if name in used_names:
        unique_name = f"{name}__duplicate_{suffix}"
        while unique_name in used_names or unique_name in reserved_names:
            suffix += 1
            unique_name = f"{name}__duplicate_{suffix}"
    used_names.add(unique_name)
    unique_header.append(unique_name)
    if unique_name != name:
        renamed_columns.append({"column_position": position + 1, "original": name, "renamed": unique_name})

normalized_header = unique_header
if renamed_columns:
    print("Repeated column names were renamed; all column values are preserved:")
    print(json.dumps(renamed_columns, indent=2))

preview = pd.read_csv(CSV_PATH, nrows=PREVIEW_ROWS, encoding="utf-8-sig", low_memory=False)
preview.columns = normalized_header
if preview.empty:
    raise ValueError("This CSV has no data rows. Select a nonempty dataset file.")

label_columns = [name for name in preview.columns if name.casefold() == "label"]
if len(label_columns) != 1:
    raise ValueError("Expected one Label column. Share the column list: " + str(preview.columns.tolist()))
label_column = label_columns[0]
labels = preview[label_column].astype("string").str.strip().replace("", pd.NA)

# Numeric coercion here measures the preview only; it is not a fitted preprocessor.
numeric_preview = preview.drop(columns=[label_column]).apply(pd.to_numeric, errors="coerce")
infinite_counts = np.isinf(numeric_preview).sum()
missing_counts = preview.isna().sum()
label_counts = labels.fillna("<MISSING>").value_counts()
valid_labels = labels.dropna().str.upper()
has_benign = bool(valid_labels.eq("BENIGN").any())
has_attack = bool(valid_labels.ne("BENIGN").any())

report = {
    "file": CSV_PATH.name,
    "file_bytes": CSV_PATH.stat().st_size,
    "scope": "first rows only; not representative training data or full-file statistics",
    "preview_rows": len(preview),
    "preview_limit": PREVIEW_ROWS,
    "column_count": len(preview.columns),
    "columns": preview.columns.tolist(),
    "renamed_duplicate_columns": renamed_columns,
    "dtypes": {name: str(dtype) for name, dtype in preview.dtypes.items()},
    "label_column": label_column,
    "preview_label_counts": {str(name): int(count) for name, count in label_counts.items()},
    "missing_labels": int(labels.isna().sum()),
    "preview_missing_values": {name: int(count) for name, count in missing_counts.items() if count},
    "preview_numeric_infinite_values": {name: int(count) for name, count in infinite_counts.items() if count},
    "preview_duplicate_rows": int(preview.duplicated().sum()),
    "both_binary_classes_in_preview": has_benign and has_attack,
    "versions": {"python": sys.version.split()[0], "pandas": pd.__version__, "numpy": np.__version__},
}

print(json.dumps(report, indent=2, ensure_ascii=False))
if not (has_benign and has_attack):
    print("\nNOTE: Both BENIGN and non-BENIGN labels are not visible in this preview.")
    print("This does not prove the full CSV has only one class. Inspect full-file counts next.")

REPORT_PATH = Path("dataset_inspection.json")
REPORT_PATH.write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding="utf-8")
print("\nSaved:", REPORT_PATH.resolve())

## 6. Count labels across the entire CSV

Run this after the inspection cell succeeds. The first 10,000 rows can miss attacks. This scans only the Label column in chunks, preserving memory, and saves full_label_counts.json. Counts precede cleaning and sampling. Send this output before we prepare the 20,000-row subset.

In [ ]:
import csv
import json
from collections import Counter
from pathlib import Path

import pandas as pd

# Locate the same CSV used in the successful inspection.
DATA_DIR = Path("/content")
inspection = json.loads(
    (DATA_DIR / "dataset_inspection.json").read_text(encoding="utf-8")
)
CSV_PATH = DATA_DIR / inspection["file"]

if not CSV_PATH.is_file():
    raise FileNotFoundError("Upload the inspected CSV again: " + str(CSV_PATH))

# Find the original Label header, including any surrounding spaces.
with CSV_PATH.open("r", encoding="utf-8-sig", newline="") as handle:
    header = next(csv.reader(handle), [])

label_headers = [name for name in header if name.strip().casefold() == "label"]
if len(label_headers) != 1:
    raise ValueError("Expected exactly one Label column.")

raw_label_column = label_headers[0]
full_label_counts = Counter()
total_rows = 0
missing_labels = 0

# Read only labels, 50,000 rows at a time. No features or models are loaded.
for chunk in pd.read_csv(
    CSV_PATH,
    usecols=[raw_label_column],
    chunksize=50_000,
    encoding="utf-8-sig",
):
    labels = chunk[raw_label_column].astype("string").str.strip().replace("", pd.NA)
    total_rows += len(labels)
    missing_labels += int(labels.isna().sum())
    full_label_counts.update(labels.dropna().tolist())

benign_rows = sum(
    count for label, count in full_label_counts.items()
    if label.casefold() == "benign"
)
attack_rows = sum(full_label_counts.values()) - benign_rows

full_report = {
    "file": CSV_PATH.name,
    "total_rows": total_rows,
    "missing_labels": missing_labels,
    "full_label_counts": dict(sorted(full_label_counts.items())),
    "benign_rows": benign_rows,
    "non_benign_rows": attack_rows,
    "both_binary_classes_present": benign_rows > 0 and attack_rows > 0,
    "scope": "Full-file label counts before cleaning or sampling",
}

print(json.dumps(full_report, indent=2))
output_path = DATA_DIR / "full_label_counts.json"
output_path.write_text(json.dumps(full_report, indent=2), encoding="utf-8")
print("\nSaved:", output_path)

if benign_rows > 0 and attack_rows > 0:
    print("Both classes are present. We can prepare the smaller dataset next.")
else:
    print("STOP: This file does not contain both BENIGN and non-BENIGN labels.")
    print("We need another labeled CSV before binary classification.")


## 7. Clean, sample and split

Run after full-file counts confirm BENIGN and DDoS. This loads the single CSV in Colab, converts numeric inputs, replaces infinities with NaN, removes repeated and conflicting feature records, and samples up to 20,000 rows with the cleaned class proportions. It reserves 60% train, 20% validation and 20% final test. Missing values remain for a training-only imputer in the next step. This random baseline does not measure generalization to a different day or network. Copy the equivalent complete code from phase1_prepare_colab.py if using an older Colab copy.

In [ ]:
%pip install -q "scikit-learn>=1.6,<2"

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
from sklearn.model_selection import train_test_split

SEED = 42
TARGET_ROWS = 20_000
ROOT = Path("/content/SENTINEL-AI")

# Load the same file and unique column names used during inspection.
inspection = json.loads(
    Path("/content/dataset_inspection.json").read_text(encoding="utf-8")
)
CSV_PATH = Path("/content") / inspection["file"]
data = pd.read_csv(CSV_PATH, encoding="utf-8-sig", low_memory=False)
if len(data.columns) != len(inspection["columns"]):
    raise ValueError("The file schema changed. Run inspection again.")
data.columns = inspection["columns"]
original_rows = len(data)

label_column = inspection["label_column"]
labels = data[label_column].astype("string").str.strip().replace("", pd.NA)
missing_labels = int(labels.isna().sum())
data = data.loc[labels.notna()].copy()
labels = labels.loc[labels.notna()].str.upper()

# This first model is specifically BENIGN versus DDoS.
if set(labels.unique()) != {"BENIGN", "DDOS"}:
    raise ValueError("Expected BENIGN and DDoS. Inspect the actual labels before continuing.")

# Exclude identifiers and timestamps from model inputs.
identifier_names = {
    "flowid", "sourceip", "srcip", "destinationip", "dstip",
    "timestamp", "time", "date",
}
excluded = [
    name for name in data.columns
    if name != label_column
    and "".join(c for c in name.lower() if c.isalnum()) in identifier_names
]
features = [
    name for name in data.columns
    if name != label_column and name not in excluded
]
if not features:
    raise ValueError("No candidate features remain.")

# Fixed conversions only: learn no averages or feature-selection rules here.
X = data[features].apply(pd.to_numeric, errors="coerce")
infinite_values = int(np.isinf(X).sum().sum())
X = X.replace([np.inf, -np.inf], np.nan)
clean = X.copy()
clean["Label"] = np.where(labels.eq("BENIGN"), "BENIGN", "ATTACK")

empty_rows = clean[features].isna().all(axis=1)
empty_feature_rows = int(empty_rows.sum())
clean = clean.loc[~empty_rows].copy()

# Remove repeated model-input/label records before any random split.
before = len(clean)
clean = clean.drop_duplicates(subset=features + ["Label"])
duplicate_rows_removed = before - len(clean)

# Remaining identical inputs with different labels are ambiguous.
# Remove both records rather than putting them in different partitions.
conflicting = clean.duplicated(subset=features, keep=False)
conflicting_rows_removed = int(conflicting.sum())
clean = clean.loc[~conflicting].copy()

if set(clean["Label"]) != {"BENIGN", "ATTACK"}:
    raise ValueError("Both classes must remain after cleaning.")

# Preserve the cleaned data's class proportions, rather than forcing 50/50.
if len(clean) > TARGET_ROWS:
    sample, _ = train_test_split(
        clean,
        train_size=TARGET_ROWS,
        stratify=clean["Label"],
        random_state=SEED,
    )
else:
    sample = clean.sample(frac=1, random_state=SEED).copy()

if sample["Label"].value_counts().min() < 5:
    raise ValueError("Too few records in one class for a reliable three-way split.")

# 60% training, 20% validation, 20% untouched final test.
train, remaining = train_test_split(
    sample, test_size=0.4, stratify=sample["Label"], random_state=SEED
)
validation, test = train_test_split(
    remaining, test_size=0.5, stratify=remaining["Label"], random_state=SEED
)

assert set(train.index).isdisjoint(validation.index)
assert set(train.index).isdisjoint(test.index)
assert set(validation.index).isdisjoint(test.index)
assert len(train) + len(validation) + len(test) == len(sample)
assert not sample.duplicated(subset=features).any()

# Missing values remain NaN. Fit the imputer using training data only later.
(ROOT / "data/sample").mkdir(parents=True, exist_ok=True)
(ROOT / "data/processed").mkdir(parents=True, exist_ok=True)
sample.to_csv(ROOT / "data/sample/sentinel_subset.csv", index=False)

splits = {"train": train, "validation": validation, "test": test}
for name, frame in splits.items():
    frame.to_csv(ROOT / f"data/processed/{name}.csv", index=False)

prep_report = {
    "source_file": CSV_PATH.name,
    "original_rows": original_rows,
    "missing_labels_removed": missing_labels,
    "infinite_values_replaced_with_nan": infinite_values,
    "empty_feature_rows_removed": empty_feature_rows,
    "duplicate_rows_removed": duplicate_rows_removed,
    "conflicting_rows_removed": conflicting_rows_removed,
    "cleaned_rows": len(clean),
    "cleaned_class_counts": clean["Label"].value_counts().to_dict(),
    "sample_rows": len(sample),
    "sample_class_counts": sample["Label"].value_counts().to_dict(),
    "excluded_identifier_columns": excluded,
    "candidate_feature_count": len(features),
    "candidate_features": features,
    "splits": {
        name: {
            "rows": len(frame),
            "class_counts": frame["Label"].value_counts().to_dict(),
        }
        for name, frame in splits.items()
    },
    "seed": SEED,
    "label_mapping": {"BENIGN": 0, "ATTACK": 1},
    "supported_attack_labels": ["DDoS"],
    "split_method": "Stratified random split within one capture file",
    "limitation": "Related flows may remain across splits; this is not an independent-day test.",
    "preprocessing": "No imputer or feature selector fitted yet; NaN values retained.",
    "versions": {"pandas": pd.__version__, "numpy": np.__version__, "sklearn": sklearn.__version__},
}

(ROOT / "preparation_report.json").write_text(
    json.dumps(prep_report, indent=2), encoding="utf-8"
)
print(json.dumps(prep_report, indent=2))
print("\nSaved data under:", ROOT)
print("No model has been trained. Do not use the test set to choose a model.")



## 8. Random Forest baseline: training and validation only

Run after Step 7 succeeds. The imputer learns medians from training data, then constant-feature removal also learns only from training. The classifier is a small Random Forest. Validation data measures the candidate; test.csv remains reserved for final selection. Raw class probabilities are not calibrated confidence scores.

The script saves candidate models and metadata under artifacts/candidates/random_forest, verifies reloaded artifacts reproduce validation predictions, and generates a validation confusion matrix. These artifacts have not been promoted as the final model. In an older Colab notebook, upload phase1_random_forest_colab.py to the Files panel and run it with %run instead of copying this cell.

Background: https://scikit-learn.org/1.6/modules/generated/sklearn.impute.SimpleImputer.html

In [ ]:
%pip install -q "matplotlib>=3.8,<4"

In [ ]:
import json
import platform
from pathlib import Path
from time import perf_counter

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import VarianceThreshold
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline

ROOT = Path("/content/SENTINEL-AI")
OUTPUT = ROOT / "artifacts/candidates/random_forest"
SEED = 42
LABEL_MAPPING = {"BENIGN": 0, "ATTACK": 1}

if not (ROOT / "preparation_report.json").is_file():
    raise FileNotFoundError("Run Step 7 first. Its preparation report is missing.")

preparation = json.loads((ROOT / "preparation_report.json").read_text(encoding="utf-8"))
feature_names = preparation["candidate_features"]
if not feature_names or len(feature_names) != len(set(feature_names)):
    raise ValueError("Candidate feature names must be nonempty and unique.")

# Read only training and validation data. Reserve test.csv for final evaluation.
train = pd.read_csv(ROOT / "data/processed/train.csv")
validation = pd.read_csv(ROOT / "data/processed/validation.csv")
expected_columns = set(feature_names) | {"Label"}
for name, frame in {"train": train, "validation": validation}.items():
    if set(frame.columns) != expected_columns:
        raise ValueError(f"{name} columns do not match the preparation report.")
    if set(frame["Label"]) != set(LABEL_MAPPING):
        raise ValueError(f"{name} must contain BENIGN and ATTACK labels only.")

X_train = train[feature_names].apply(pd.to_numeric, errors="raise")
X_validation = validation[feature_names].apply(pd.to_numeric, errors="raise")
X_train = X_train.replace([np.inf, -np.inf], np.nan)
X_validation = X_validation.replace([np.inf, -np.inf], np.nan)
y_train = train["Label"].map(LABEL_MAPPING).to_numpy(dtype=int)
y_validation = validation["Label"].map(LABEL_MAPPING).to_numpy(dtype=int)

# Fit missing-value handling and constant-feature removal on training data only.
# Empty training columns become zero and are removed by VarianceThreshold.
preprocessor = Pipeline([
    ("imputer", SimpleImputer(strategy="median", keep_empty_features=True)),
    ("constant_filter", VarianceThreshold(threshold=0.0)),
])

model = RandomForestClassifier(
    n_estimators=100,
    max_depth=16,
    min_samples_leaf=2,
    class_weight="balanced_subsample",
    random_state=SEED,
    n_jobs=2,
)

print("Training Random Forest; the test set remains reserved.")
started = perf_counter()
train_prepared = preprocessor.fit_transform(X_train)
model.fit(train_prepared, y_train)
training_seconds = perf_counter() - started

started = perf_counter()
validation_prepared = preprocessor.transform(X_validation)
probabilities = model.predict_proba(validation_prepared)
predictions = model.classes_[probabilities.argmax(axis=1)]
prediction_seconds = perf_counter() - started
attack_probabilities = probabilities[:, list(model.classes_).index(1)]

matrix = confusion_matrix(y_validation, predictions, labels=[0, 1])
tn, fp, fn, tp = matrix.ravel()
metrics = {
    "model": "RandomForestClassifier",
    "candidate_version": "rf-baseline-001",
    "evaluation_split": "validation",
    "training_rows": len(train),
    "validation_rows": len(validation),
    "input_feature_count": len(feature_names),
    "selected_feature_count": int(train_prepared.shape[1]),
    "accuracy": float(accuracy_score(y_validation, predictions)),
    "attack_precision": float(precision_score(y_validation, predictions, zero_division=0)),
    "attack_recall": float(recall_score(y_validation, predictions, zero_division=0)),
    "attack_f1": float(f1_score(y_validation, predictions, zero_division=0)),
    "false_positive_rate": float(fp / (fp + tn)),
    "roc_auc": float(roc_auc_score(y_validation, attack_probabilities)),
    "confusion_matrix": matrix.tolist(),
    "confusion_matrix_order": ["BENIGN", "ATTACK"],
    "training_seconds": training_seconds,
    "validation_batch_ms_per_flow": prediction_seconds * 1000 / len(validation),
    "latency_note": "Amortized batch time including preprocessing; not single-request API latency.",
    "probability_note": "Random Forest probabilities are not calibrated confidence scores.",
    "classification_report": classification_report(
        y_validation, predictions, labels=[0, 1],
        target_names=["BENIGN", "ATTACK"], output_dict=True, zero_division=0,
    ),
}

# Candidate artifacts only. Final model selection follows XGBoost comparison.
OUTPUT.mkdir(parents=True, exist_ok=True)
joblib.dump(model, OUTPUT / "model.joblib", compress=3)
joblib.dump(preprocessor, OUTPUT / "preprocessor.joblib", compress=3)
metrics["model_bytes"] = (OUTPUT / "model.joblib").stat().st_size

selected_names = np.asarray(feature_names)[
    preprocessor.named_steps["constant_filter"].get_support()
].tolist()
schema = {
    "feature_names": feature_names,
    "feature_types": {name: "numeric" for name in feature_names},
    "feature_order_required": True,
    "missing_values": "NaN accepted; training-fitted median imputation applied.",
    "selected_feature_names": selected_names,
    "supported_attack_labels": ["DDoS"],
    "label_mapping": LABEL_MAPPING,
}
metadata = {
    "candidate_version": "rf-baseline-001",
    "model_params": model.get_params(),
    "preprocessing": {"imputer": "training median", "constant_filter_threshold": 0.0},
    "seed": SEED,
    "source_file": preparation["source_file"],
    "versions": {
        "python": platform.python_version(), "numpy": np.__version__,
        "pandas": pd.__version__, "scikit-learn": sklearn.__version__, "joblib": joblib.__version__,
    },
}
for filename, content in {
    "feature_schema.json": schema,
    "model_params.json": metadata,
    "label_mapping.json": LABEL_MAPPING,
    "evaluation_metrics.json": metrics,
}.items():
    (OUTPUT / filename).write_text(json.dumps(content, indent=2), encoding="utf-8")

# Verify saved preprocessing and model reproduce predictions on validation rows.
loaded_preprocessor = joblib.load(OUTPUT / "preprocessor.joblib")
loaded_model = joblib.load(OUTPUT / "model.joblib")
reloaded_predictions = loaded_model.predict(loaded_preprocessor.transform(X_validation.iloc[:10]))
assert np.array_equal(reloaded_predictions, predictions[:10])

print(json.dumps(metrics, indent=2))
print("\nSaved candidate artifacts:", OUTPUT)
print("Artifact reload check passed. Final test evaluation has not run.")

display = ConfusionMatrixDisplay(matrix, display_labels=["BENIGN", "ATTACK"])
display.plot(cmap="Blues", values_format="d")
plt.title("Random Forest — validation confusion matrix")
plt.tight_layout()
plt.savefig(OUTPUT / "validation_confusion_matrix.png", dpi=150)
plt.show()


## 9. XGBoost and validation comparison

Run after the Random Forest baseline. Reuse its training-fitted preprocessing and the same splits. The script checks that the saved baseline still reproduces its validation confusion matrix before comparing it with XGBoost. Test data remains reserved.

Educational validation targets are attack recall >= 98%, attack F1 >= 98%, false positive rate <= 1%. Among passing candidates prioritize recall, then lower FPR, higher F1, smaller model size, lower batch latency. The report recommends a candidate; it does not deploy a model or report final test metrics.

The XGBoost candidate is saved in joblib and native UBJSON formats; see https://xgboost.readthedocs.io/en/stable/tutorials/saving_model.html. Copy the complete phase1_xgboost_colab.py file into the runtime and use %run if working in an older Colab notebook.

In [ ]:
%pip install -q "xgboost>=2.1,<4"

In [ ]:
import json
import platform
from pathlib import Path
from time import perf_counter

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
import xgboost
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from xgboost import XGBClassifier

ROOT = Path("/content/SENTINEL-AI")
RF_OUTPUT = ROOT / "artifacts/candidates/random_forest"
OUTPUT = ROOT / "artifacts/candidates/xgboost"
SEED = 42
LABEL_MAPPING = {"BENIGN": 0, "ATTACK": 1}

# Set these educational validation targets before running the new candidate.
QUALITY_GATE = {
    "min_attack_recall": 0.98,
    "min_attack_f1": 0.98,
    "max_false_positive_rate": 0.01,
}

required = [
    ROOT / "preparation_report.json",
    RF_OUTPUT / "model.joblib",
    RF_OUTPUT / "preprocessor.joblib",
    RF_OUTPUT / "feature_schema.json",
    RF_OUTPUT / "evaluation_metrics.json",
]
if any(not path.is_file() for path in required):
    raise FileNotFoundError("Run Steps 7 and 8 first; preparation or Random Forest artifacts are missing.")

preparation = json.loads((ROOT / "preparation_report.json").read_text(encoding="utf-8"))
schema = json.loads((RF_OUTPUT / "feature_schema.json").read_text(encoding="utf-8"))
rf_metrics = json.loads((RF_OUTPUT / "evaluation_metrics.json").read_text(encoding="utf-8"))
feature_names = preparation["candidate_features"]
if schema["feature_names"] != feature_names:
    raise ValueError("The Random Forest schema differs from the prepared dataset. Rerun Step 8.")

train = pd.read_csv(ROOT / "data/processed/train.csv")
validation = pd.read_csv(ROOT / "data/processed/validation.csv")
for name, frame in {"train": train, "validation": validation}.items():
    if set(frame.columns) != set(feature_names) | {"Label"}:
        raise ValueError(f"{name} columns do not match the feature schema.")
    if set(frame["Label"]) != set(LABEL_MAPPING):
        raise ValueError(f"{name} must contain both BENIGN and ATTACK labels only.")
if rf_metrics["training_rows"] != len(train) or rf_metrics["validation_rows"] != len(validation):
    raise ValueError("Split sizes changed since Step 8. Rerun the baseline on the current splits.")

X_train = train[feature_names].apply(pd.to_numeric, errors="raise").replace([np.inf, -np.inf], np.nan)
X_validation = validation[feature_names].apply(pd.to_numeric, errors="raise").replace([np.inf, -np.inf], np.nan)
y_train = train["Label"].map(LABEL_MAPPING).to_numpy(dtype=int)
y_validation = validation["Label"].map(LABEL_MAPPING).to_numpy(dtype=int)

# Reuse the exact preprocessing already fitted on training data in Step 8.
# Never refit it on validation data. No test.csv is read anywhere in this step.
preprocessor = joblib.load(RF_OUTPUT / "preprocessor.joblib")
train_prepared = preprocessor.transform(X_train)
validation_prepared = preprocessor.transform(X_validation)
rf_model = joblib.load(RF_OUTPUT / "model.joblib")
rf_predictions = rf_model.predict(validation_prepared)
rf_matrix = confusion_matrix(y_validation, rf_predictions, labels=[0, 1])
if rf_matrix.tolist() != rf_metrics["confusion_matrix"]:
    raise ValueError("The saved baseline no longer reproduces its validation results. Rerun Step 8.")

params = {
    "n_estimators": 150,
    "max_depth": 4,
    "learning_rate": 0.1,
    "subsample": 0.9,
    "colsample_bytree": 0.9,
    "objective": "binary:logistic",
    "eval_metric": "logloss",
    "tree_method": "hist",
    "random_state": SEED,
    "n_jobs": 2,
}
model = XGBClassifier(**params)
print("Training XGBoost on the same training split; final test data stays reserved.")
started = perf_counter()
model.fit(train_prepared, y_train)
training_seconds = perf_counter() - started

started = perf_counter()
validation_prepared = preprocessor.transform(X_validation)
probabilities = model.predict_proba(validation_prepared)
predictions = model.classes_[probabilities.argmax(axis=1)]
prediction_seconds = perf_counter() - started
attack_probabilities = probabilities[:, list(model.classes_).index(1)]
matrix = confusion_matrix(y_validation, predictions, labels=[0, 1])
tn, fp, fn, tp = matrix.ravel()

metrics = {
    "model": "XGBClassifier",
    "candidate_version": "xgb-baseline-001",
    "evaluation_split": "validation",
    "training_rows": len(train),
    "validation_rows": len(validation),
    "input_feature_count": len(feature_names),
    "selected_feature_count": int(train_prepared.shape[1]),
    "accuracy": float(accuracy_score(y_validation, predictions)),
    "attack_precision": float(precision_score(y_validation, predictions, zero_division=0)),
    "attack_recall": float(recall_score(y_validation, predictions, zero_division=0)),
    "attack_f1": float(f1_score(y_validation, predictions, zero_division=0)),
    "false_positive_rate": float(fp / (fp + tn)),
    "roc_auc": float(roc_auc_score(y_validation, attack_probabilities)),
    "confusion_matrix": matrix.tolist(),
    "confusion_matrix_order": ["BENIGN", "ATTACK"],
    "training_seconds": training_seconds,
    "training_time_note": "Classifier fit only; preprocessing was reused from Random Forest.",
    "validation_batch_ms_per_flow": prediction_seconds * 1000 / len(validation),
    "latency_note": "Amortized batch time including preprocessing; not single-request API latency.",
    "probability_note": "XGBoost probabilities are not calibrated confidence scores.",
    "classification_report": classification_report(
        y_validation, predictions, labels=[0, 1],
        target_names=["BENIGN", "ATTACK"], output_dict=True, zero_division=0,
    ),
}

OUTPUT.mkdir(parents=True, exist_ok=True)
joblib.dump(model, OUTPUT / "model.joblib", compress=3)
joblib.dump(preprocessor, OUTPUT / "preprocessor.joblib", compress=3)
model.save_model(OUTPUT / "model.ubj")
metrics["model_bytes"] = (OUTPUT / "model.joblib").stat().st_size
metrics["native_model_bytes"] = (OUTPUT / "model.ubj").stat().st_size
metadata = {
    "candidate_version": "xgb-baseline-001",
    "model_params": params,
    "preprocessing": "Unchanged training-fitted pipeline from the Random Forest baseline.",
    "seed": SEED,
    "source_file": preparation["source_file"],
    "versions": {
        "python": platform.python_version(), "numpy": np.__version__,
        "pandas": pd.__version__, "scikit-learn": sklearn.__version__,
        "joblib": joblib.__version__, "xgboost": xgboost.__version__,
    },
}
for filename, content in {
    "feature_schema.json": schema,
    "model_params.json": metadata,
    "label_mapping.json": LABEL_MAPPING,
    "evaluation_metrics.json": metrics,
}.items():
    (OUTPUT / filename).write_text(json.dumps(content, indent=2), encoding="utf-8")

loaded_preprocessor = joblib.load(OUTPUT / "preprocessor.joblib")
loaded_model = joblib.load(OUTPUT / "model.joblib")
reloaded_predictions = loaded_model.predict(loaded_preprocessor.transform(X_validation.iloc[:10]))
assert np.array_equal(reloaded_predictions, predictions[:10])
native_model = XGBClassifier(n_jobs=2)
native_model.load_model(OUTPUT / "model.ubj")
assert np.array_equal(native_model.predict(validation_prepared[:10]), predictions[:10])

# Compare measured validation results; these targets are educational design choices.
rows = []
for name, result in {"random_forest": rf_metrics, "xgboost": metrics}.items():
    rows.append({
        "candidate": name,
        **{key: result[key] for key in [
            "accuracy", "attack_precision", "attack_recall", "attack_f1",
            "false_positive_rate", "roc_auc", "model_bytes", "validation_batch_ms_per_flow",
        ]},
        "passes_quality_gate": bool(
            result["attack_recall"] >= QUALITY_GATE["min_attack_recall"]
            and result["attack_f1"] >= QUALITY_GATE["min_attack_f1"]
            and result["false_positive_rate"] <= QUALITY_GATE["max_false_positive_rate"]
        ),
    })

comparison = pd.DataFrame(rows)
eligible = comparison.loc[comparison["passes_quality_gate"]].sort_values(
    ["attack_recall", "false_positive_rate", "attack_f1", "model_bytes", "validation_batch_ms_per_flow"],
    ascending=[False, True, False, True, True], kind="stable",
)
recommended = None if eligible.empty else str(eligible.iloc[0]["candidate"])
summary = {
    "evaluation_split": "validation",
    "quality_gate": QUALITY_GATE,
    "ranking": "Among passing candidates: recall descending, FPR ascending, F1 descending, size ascending, latency ascending.",
    "candidates": rows,
    "recommended_candidate": recommended,
    "status": "Recommendation only; final test evaluation and promotion have not run.",
}
comparison.to_csv(ROOT / "artifacts/model_comparison.csv", index=False)
(ROOT / "artifacts/model_comparison.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")

print(json.dumps(metrics, indent=2))
print("\nVALIDATION COMPARISON (metric values are fractions, not percentages)")
print(comparison.to_string(index=False))
print("\nValidation recommendation:", recommended or "No candidate passed the quality gate")
print("Saved XGBoost artifacts:", OUTPUT)
print("Both reload checks passed. Final test evaluation has not run.")
display = ConfusionMatrixDisplay(matrix, display_labels=["BENIGN", "ATTACK"])
display.plot(cmap="Blues", values_format="d")
plt.title("XGBoost — validation confusion matrix")
plt.tight_layout()
plt.savefig(OUTPUT / "validation_confusion_matrix.png", dpi=150)
plt.show()


## 10. Separate anomaly detection with Isolation Forest

Step 9's validation recommendation is XGBoost. This step adds a separate anomaly detector rather than assigning attack categories. Reuse the selected classifier's training-fitted preprocessing, then fit Isolation Forest using known BENIGN training flows as the normal reference. Validation data is scored, not used for fitting; test.csv stays reserved.

The fixed contamination setting 0.05 defines a training-reference cutoff. It is not a guaranteed validation false-positive rate or a probability of attack. Report -decision_function, so higher scores are more unusual and >0 indicates an anomaly. Diagnostic metrics compare these flags to known validation labels without claiming category classification.

Documentation: https://scikit-learn.org/1.6/modules/generated/sklearn.ensemble.IsolationForest.html

In an older Colab notebook upload phase1_isolation_forest_colab.py through the Files panel and run it with %run.

In [ ]:
import json
import platform
from pathlib import Path
from time import perf_counter

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from sklearn.ensemble import IsolationForest
from sklearn.metrics import confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score

ROOT = Path("/content/SENTINEL-AI")
OUTPUT = ROOT / "artifacts/candidates/isolation_forest"
SEED = 42
CONTAMINATION = 0.05

comparison_path = ROOT / "artifacts/model_comparison.json"
if not comparison_path.is_file():
    raise FileNotFoundError("Run Step 9 first. The model comparison report is missing.")
comparison = json.loads(comparison_path.read_text(encoding="utf-8"))
selected = comparison["recommended_candidate"]
if selected not in {"random_forest", "xgboost"}:
    raise ValueError("No supervised candidate was recommended. Review validation results first.")
if not any(row["candidate"] == selected and row["passes_quality_gate"] for row in comparison["candidates"]):
    raise ValueError("The recommended classifier must pass the validation quality gate.")

candidate = ROOT / "artifacts/candidates" / selected
schema = json.loads((candidate / "feature_schema.json").read_text(encoding="utf-8"))
features = schema["feature_names"]
preprocessor = joblib.load(candidate / "preprocessor.joblib")

# Reuse saved training-only preprocessing. Do not fit it again.
train = pd.read_csv(ROOT / "data/processed/train.csv")
validation = pd.read_csv(ROOT / "data/processed/validation.csv")
for name, frame in {"train": train, "validation": validation}.items():
    if set(frame.columns) != set(features) | {"Label"}:
        raise ValueError(f"{name} columns do not match the selected classifier schema.")
    if set(frame["Label"]) != {"BENIGN", "ATTACK"}:
        raise ValueError(f"{name} must contain both BENIGN and ATTACK labels only.")

# Known BENIGN training records define the normal reference distribution.
# Isolation Forest receives features only, not attack labels as training targets.
normal_reference = train.loc[train["Label"].eq("BENIGN"), features]
if len(normal_reference) < 20:
    raise ValueError("Too few BENIGN training records for the anomaly reference.")
normal_reference = normal_reference.apply(pd.to_numeric, errors="raise").replace([np.inf, -np.inf], np.nan)
X_validation = validation[features].apply(pd.to_numeric, errors="raise").replace([np.inf, -np.inf], np.nan)
reference_prepared = preprocessor.transform(normal_reference)
validation_prepared = preprocessor.transform(X_validation)

params = {
    "n_estimators": 100,
    "max_samples": min(256, len(normal_reference)),
    "contamination": CONTAMINATION,
    "random_state": SEED,
    "n_jobs": 2,
}
anomaly_model = IsolationForest(**params)
started = perf_counter()
anomaly_model.fit(reference_prepared)
training_seconds = perf_counter() - started

# Negating decision_function makes larger scores mean more unusual flows.
# A score greater than zero corresponds to Isolation Forest's -1 anomaly flag.
anomaly_scores = -anomaly_model.decision_function(validation_prepared)
anomaly_flags = anomaly_model.predict(validation_prepared) == -1
assert np.array_equal(anomaly_flags, anomaly_scores > 0)
y_validation = validation["Label"].map({"BENIGN": 0, "ATTACK": 1}).to_numpy(dtype=int)
matrix = confusion_matrix(y_validation, anomaly_flags.astype(int), labels=[0, 1])
tn, fp, fn, tp = matrix.ravel()

metrics = {
    "model": "IsolationForest",
    "candidate_version": "if-baseline-001",
    "evaluation_split": "validation",
    "training_reference": "BENIGN records from the training split only",
    "training_reference_rows": len(normal_reference),
    "validation_rows": len(validation),
    "selected_classifier": selected,
    "contamination": CONTAMINATION,
    "threshold_note": "5% is a training reference cutoff setting, not the live attack probability or guaranteed validation false-positive rate.",
    "anomaly_score_definition": "-decision_function; higher is more unusual; > 0 flags an anomaly",
    "score_is_probability": False,
    "predicts_attack_category": False,
    "validation_anomaly_count": int(anomaly_flags.sum()),
    "validation_anomaly_fraction": float(anomaly_flags.mean()),
    "training_seconds": training_seconds,
    "diagnostics_against_known_binary_labels": {
        "note": "Compare anomaly flags with known validation labels for diagnosis only; these flags do not assign attack categories.",
        "precision": float(precision_score(y_validation, anomaly_flags, zero_division=0)),
        "attack_recall": float(recall_score(y_validation, anomaly_flags, zero_division=0)),
        "f1": float(f1_score(y_validation, anomaly_flags, zero_division=0)),
        "benign_false_positive_rate": float(fp / (tn + fp)),
        "roc_auc": float(roc_auc_score(y_validation, anomaly_scores)),
        "confusion_matrix": matrix.tolist(),
        "matrix_true_rows": ["BENIGN", "ATTACK"],
        "matrix_predicted_columns": ["NORMAL_FLAG", "ANOMALY_FLAG"],
    },
}

OUTPUT.mkdir(parents=True, exist_ok=True)
joblib.dump(anomaly_model, OUTPUT / "anomaly_model.joblib", compress=3)
joblib.dump(preprocessor, OUTPUT / "preprocessor.joblib", compress=3)
metrics["model_bytes"] = (OUTPUT / "anomaly_model.joblib").stat().st_size
metadata = {
    "candidate_version": "if-baseline-001",
    "model_params": params,
    "anomaly_score_definition": metrics["anomaly_score_definition"],
    "preprocessing": "Shared unchanged training-fitted pipeline from the selected classifier.",
    "versions": {
        "python": platform.python_version(), "numpy": np.__version__,
        "pandas": pd.__version__, "scikit-learn": sklearn.__version__, "joblib": joblib.__version__,
    },
}
for filename, content in {
    "feature_schema.json": schema,
    "model_params.json": metadata,
    "evaluation_metrics.json": metrics,
}.items():
    (OUTPUT / filename).write_text(json.dumps(content, indent=2), encoding="utf-8")

pd.DataFrame({
    "validation_row": np.arange(len(validation)),
    "true_binary_label": validation["Label"],
    "anomaly_score": anomaly_scores,
    "anomaly_flag": anomaly_flags,
}).to_csv(OUTPUT / "validation_anomaly_scores.csv", index=False)

loaded_model = joblib.load(OUTPUT / "anomaly_model.joblib")
loaded_preprocessor = joblib.load(OUTPUT / "preprocessor.joblib")
reloaded_scores = -loaded_model.decision_function(loaded_preprocessor.transform(X_validation.iloc[:10]))
assert np.allclose(reloaded_scores, anomaly_scores[:10])
assert np.array_equal(loaded_model.predict(validation_prepared[:10]) == -1, anomaly_flags[:10])

print(json.dumps(metrics, indent=2))
print("\nSaved anomaly artifacts:", OUTPUT)
print("Reload checks passed. Classifier recommendation remains:", selected)
print("Anomaly flags are separate from attack classification. No test data was evaluated.")

fig, ax = plt.subplots()
for label in ["BENIGN", "ATTACK"]:
    ax.hist(anomaly_scores[validation["Label"].eq(label).to_numpy()], bins=35, alpha=0.55, label=label)
ax.axvline(0, color="black", linestyle="--", label="Anomaly cutoff")
ax.set_title("Isolation Forest — validation score distributions")
ax.set_xlabel("Anomaly score (not a probability); higher is more unusual")
ax.set_ylabel("Number of validation flows")
ax.legend()
fig.tight_layout()
fig.savefig(OUTPUT / "validation_anomaly_scores.png", dpi=150)
plt.show()


## 11. Final test evaluation and export

Run after the validation comparison and separate anomaly detector are complete. Freeze the classifier choice and original quality thresholds before inspecting test labels. Do not train, tune, or switch models using final test results. The script rejects changed frozen artifacts and uses saved results on unchanged reruns. Preserve its lock and test result files.

This exports the required classifier, preprocessing, anomaly model, schema, label mapping, model params and measured metrics. It verifies exported inference on historical validation examples and bundles reports, plots, package versions, demo flows and all small dataset splits. It sets eligible_for_api_demo=false if the fixed final gate fails.

Download /content/SENTINEL-AI/sentinel_phase1_bundle.zip, save the notebook, and extract the ZIP into the local project. Share the final test output and confirm the download before Phase 2. In an older Colab notebook, upload phase1_finalize_colab.py through the Files panel and run it with %run.

In [ ]:
import hashlib
import json
import platform
import shutil
from pathlib import Path
from time import perf_counter

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

ROOT = Path("/content/SENTINEL-AI")
LABEL_MAPPING = {"BENIGN": 0, "ATTACK": 1}


def read_json(path):
    return json.loads(path.read_text(encoding="utf-8"))


def write_json(path, content):
    path.write_text(json.dumps(content, indent=2, allow_nan=False), encoding="utf-8")


def fingerprint(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def main():
    artifacts = ROOT / "artifacts"
    comparison_path = artifacts / "model_comparison.json"
    if not comparison_path.is_file():
        raise FileNotFoundError("Run Steps 7-10 first; the comparison report is missing.")
    comparison = read_json(comparison_path)
    selected = comparison["recommended_candidate"]
    if selected not in {"random_forest", "xgboost"}:
        raise ValueError("No classifier passed the validation quality gate.")
    if not any(row["candidate"] == selected and row["passes_quality_gate"] for row in comparison["candidates"]):
        raise ValueError("The selected classifier must pass the original validation gate.")

    candidate = artifacts / "candidates" / selected
    anomaly = artifacts / "candidates/isolation_forest"
    required = [candidate / name for name in [
        "model.joblib", "preprocessor.joblib", "feature_schema.json",
        "model_params.json", "label_mapping.json", "evaluation_metrics.json",
    ]] + [anomaly / name for name in [
        "anomaly_model.joblib", "preprocessor.joblib", "model_params.json", "evaluation_metrics.json",
    ]]
    required += [ROOT / "preparation_report.json"]
    required += [ROOT / f"data/processed/{name}.csv" for name in ["train", "validation", "test"]]
    if selected == "xgboost":
        required.append(candidate / "model.ubj")
    if any(not path.is_file() for path in required):
        raise FileNotFoundError("A required split or saved artifact is missing. Complete Steps 7-10.")
    if fingerprint(candidate / "preprocessor.joblib") != fingerprint(anomaly / "preprocessor.joblib"):
        raise ValueError("Classifier and anomaly preprocessing differ. Recreate Step 10 before final evaluation.")

    # Freeze classifier/artifact selection before inspecting any test labels.
    lock_path = artifacts / "selection_lock.json"
    identity = {
        "selected_candidate": selected,
        "quality_gate": comparison["quality_gate"],
        "comparison_sha256": fingerprint(comparison_path),
        "artifact_sha256": {
            str(path.relative_to(ROOT)): fingerprint(path)
            for path in required if path.suffix != ".csv"
        },
    }
    if lock_path.exists():
        if read_json(lock_path) != identity:
            raise ValueError("Selection or artifacts changed after freezing. Keep the original held-out evaluation; do not switch models using its results.")
    else:
        with lock_path.open("x", encoding="utf-8") as handle:
            json.dump(identity, handle, indent=2)

    model = joblib.load(candidate / "model.joblib")
    preprocessor = joblib.load(candidate / "preprocessor.joblib")
    anomaly_model = joblib.load(anomaly / "anomaly_model.joblib")
    schema = read_json(candidate / "feature_schema.json")
    features = schema["feature_names"]
    test_path = ROOT / "data/processed/test.csv"
    test_digest = fingerprint(test_path)
    result_path = artifacts / "final_test_metrics.json"
    started_path = artifacts / "final_test_started.json"

    if result_path.exists():
        result = read_json(result_path)
        if result["test_csv_sha256"] != test_digest:
            raise ValueError("The final test file changed. Cached results belong to the original holdout.")
        print("Reusing saved final test results; no repeat test prediction is performed.")
    else:
        if started_path.exists():
            raise RuntimeError("A previous final evaluation started but did not finish. Preserve this state and share the error before retrying.")
        test = pd.read_csv(test_path)
        if set(test.columns) != set(features) | {"Label"}:
            raise ValueError("Test columns differ from the saved feature schema.")
        if set(test["Label"]) != set(LABEL_MAPPING):
            raise ValueError("Test data must contain both BENIGN and ATTACK labels only.")
        X_test = test[features].apply(pd.to_numeric, errors="raise").replace([np.inf, -np.inf], np.nan)
        y_test = test["Label"].map(LABEL_MAPPING).to_numpy(dtype=int)
        write_json(started_path, {"selected_candidate": selected, "test_csv_sha256": test_digest})
        started = perf_counter()
        prepared = preprocessor.transform(X_test)
        probabilities = model.predict_proba(prepared)
        predictions = model.classes_[probabilities.argmax(axis=1)]
        elapsed = perf_counter() - started
        attack_probabilities = probabilities[:, list(model.classes_).index(1)]
        matrix = confusion_matrix(y_test, predictions, labels=[0, 1])
        tn, fp, fn, tp = matrix.ravel()
        result = {
            "selected_candidate": selected,
            "evaluation_split": "test",
            "test_rows": len(test),
            "test_csv_sha256": test_digest,
            "accuracy": float(accuracy_score(y_test, predictions)),
            "attack_precision": float(precision_score(y_test, predictions, zero_division=0)),
            "attack_recall": float(recall_score(y_test, predictions, zero_division=0)),
            "attack_f1": float(f1_score(y_test, predictions, zero_division=0)),
            "false_positive_rate": float(fp / (fp + tn)),
            "roc_auc": float(roc_auc_score(y_test, attack_probabilities)),
            "confusion_matrix": matrix.tolist(),
            "confusion_matrix_order": ["BENIGN", "ATTACK"],
            "test_batch_ms_per_flow": elapsed * 1000 / len(test),
            "latency_note": "Amortized batch prediction time including preprocessing, not API request latency.",
            "classification_report": classification_report(
                y_test, predictions, labels=[0, 1], target_names=["BENIGN", "ATTACK"],
                output_dict=True, zero_division=0,
            ),
            "limitation": "Random holdout within one capture; results do not establish generalization to new days, networks, or attack types.",
        }
        gate = identity["quality_gate"]
        result["passes_fixed_quality_gate"] = bool(
            result["attack_recall"] >= gate["min_attack_recall"]
            and result["attack_f1"] >= gate["min_attack_f1"]
            and result["false_positive_rate"] <= gate["max_false_positive_rate"]
        )
        write_json(result_path, result)

    # Export the selected, unchanged models. No retraining or threshold tuning here.
    export = ROOT / "phase1_export"
    exported_artifacts = export / "artifacts"
    reports = export / "reports"
    samples = export / "data/sample"
    processed = export / "data/processed"
    for directory in [exported_artifacts, reports, samples, processed]:
        directory.mkdir(parents=True, exist_ok=True)
    for name in ["model.joblib", "preprocessor.joblib", "feature_schema.json", "label_mapping.json"]:
        shutil.copy2(candidate / name, exported_artifacts / name)
    shutil.copy2(anomaly / "anomaly_model.joblib", exported_artifacts / "anomaly_model.joblib")
    if (candidate / "model.ubj").exists():
        shutil.copy2(candidate / "model.ubj", exported_artifacts / "model.ubj")

    model_metadata = read_json(candidate / "model_params.json")
    anomaly_metadata = read_json(anomaly / "model_params.json")
    model_version = f"sentinel-{selected}-1.0.0"
    metadata = {
        **model_metadata,
        "model_version": model_version,
        "selected_candidate": selected,
        "selection_basis": "Validation comparison, frozen before final test labels were inspected.",
        "quality_gate": identity["quality_gate"],
        "eligible_for_api_demo": result["passes_fixed_quality_gate"],
        "anomaly_model": anomaly_metadata,
    }
    write_json(exported_artifacts / "model_params.json", metadata)
    write_json(exported_artifacts / "evaluation_metrics.json", {
        "validation": read_json(candidate / "evaluation_metrics.json"),
        "test": result,
        "anomaly_validation": read_json(anomaly / "evaluation_metrics.json"),
    })
    for path in [comparison_path, lock_path, result_path, ROOT / "preparation_report.json"]:
        shutil.copy2(path, reports / path.name)
    for name in ["train", "validation", "test"]:
        shutil.copy2(ROOT / f"data/processed/{name}.csv", processed / f"{name}.csv")
    for name in ["random_forest", "xgboost", "isolation_forest"]:
        source = artifacts / "candidates" / name
        for filename in ["evaluation_metrics.json", "validation_confusion_matrix.png", "validation_anomaly_scores.png"]:
            if (source / filename).is_file():
                shutil.copy2(source / filename, reports / f"{name}_{filename}")
    if (artifacts / "model_comparison.csv").is_file():
        shutil.copy2(artifacts / "model_comparison.csv", reports / "model_comparison.csv")

    # Demo examples come from validation, not the reserved test set.
    validation = pd.read_csv(ROOT / "data/processed/validation.csv")
    demo = pd.concat([
        validation.loc[validation["Label"].eq(label)].head(5)
        for label in ["BENIGN", "ATTACK"]
    ], ignore_index=True)
    demo.to_csv(samples / "historical_demo_flows.csv", index=False)
    X_demo = demo[features].apply(pd.to_numeric, errors="raise").replace([np.inf, -np.inf], np.nan)
    loaded_model = joblib.load(exported_artifacts / "model.joblib")
    loaded_preprocessor = joblib.load(exported_artifacts / "preprocessor.joblib")
    loaded_anomaly = joblib.load(exported_artifacts / "anomaly_model.joblib")
    demo_prepared = loaded_preprocessor.transform(X_demo)
    demo_predictions = loaded_model.predict(demo_prepared)
    demo_probabilities = loaded_model.predict_proba(demo_prepared)
    demo_scores = -loaded_anomaly.decision_function(demo_prepared)
    assert np.array_equal(demo_predictions, model.predict(preprocessor.transform(X_demo)))
    assert np.allclose(demo_scores, -anomaly_model.decision_function(preprocessor.transform(X_demo)))
    demo_records = X_demo.astype(object).where(X_demo.notna(), None).to_dict(orient="records")
    inverse_labels = {value: key for key, value in LABEL_MAPPING.items()}
    write_json(reports / "inference_smoke_test.json", {
        "source": "Historical validation examples; no live traffic or invented predictions.",
        "model_version": model_version,
        "artifact_reload_passed": True,
        "flows": [
            {
                "features": flow,
                "true_binary_label": str(demo.iloc[index]["Label"]),
                "predicted_class": inverse_labels[int(demo_predictions[index])],
                "attack_probability_uncalibrated": float(demo_probabilities[index, list(loaded_model.classes_).index(1)]),
                "anomaly_score": float(demo_scores[index]),
                "anomaly_flag": bool(demo_scores[index] > 0),
            }
            for index, flow in enumerate(demo_records)
        ],
    })

    versions = model_metadata["versions"]
    requirements = [f"{name}=={version}" for name, version in versions.items() if name != "python"]
    (export / "requirements-model.txt").write_text("\n".join(requirements) + "\n", encoding="utf-8")
    status = "PASSED" if result["passes_fixed_quality_gate"] else "FAILED; review results before API use"
    (export / "README.txt").write_text(
        f"SENTINEL AI Phase 1 export\nModel: {model_version}\nFixed quality gate: {status}\n"
        f"Training Python: {versions['python']}\nExport Python: {platform.python_version()}\n"
        "Install matching package versions from requirements-model.txt. Preserve the training Python version where feasible.\n"
        "Extract artifacts/ into the local project. Keep reports/ and data/ for the college demonstration and monitoring.\n"
        "Demo CSV rows are historical validation data. Predictions are measured, not guaranteed correct.\n"
        "Classifier is binary BENIGN/ATTACK, trained on BENIGN and DDoS only.\n"
        "Anomaly scores are separate and do not identify attack categories or calibrated probabilities.\n"
        "This is an educational prototype, not production network protection.\n", encoding="utf-8",
    )

    matrix = np.asarray(result["confusion_matrix"])
    display = ConfusionMatrixDisplay(matrix, display_labels=["BENIGN", "ATTACK"])
    display.plot(cmap="Blues", values_format="d")
    plt.title(f"{selected} — final test confusion matrix")
    plt.tight_layout()
    plt.savefig(reports / "final_test_confusion_matrix.png", dpi=150)
    plt.show()

    archive = Path(shutil.make_archive(str(ROOT / "sentinel_phase1_bundle"), "zip", export))
    print(json.dumps(result, indent=2))
    print("\nExported model and anomaly reload checks passed on historical validation examples.")
    print("Fixed quality gate:", status)
    print("Download this bundle from Colab's Files panel:", archive)
    print("Phase 2 has not started. Share the final test report before continuing.")


if __name__ == "__main__":
    main()


## Checkpoint: Phase 1 complete and verified locally

Final selected XGBoost test matrix: [[1705, 0], [6, 2289]], accuracy 99.85%, attack precision 100%, recall approximately 99.7386%, F1 approximately 99.8691%, ROC-AUC 0.999589953935305. Fixed quality gate passed. No selection or tuning used these test results.

The ZIP is downloaded and extracted into the local project. All seven artifacts and native model.ubj are present. Local Python 3.13.15 with exact exported packages reproduces all 10 historical validation predictions, probabilities, anomaly scores and flags. No final test prediction was repeated. See reports/local_verification.json and CHECKPOINT.md.

The classifier joblib snapshot fails to load on this Windows runtime; native XGBoost UBJSON loads successfully and matches Colab outputs. Preprocessing and anomaly joblib files load correctly. Future backend loading should use model.ubj for XGBoost.

Phase 1 is complete. Wait for the user's confirmation before starting Phase 2 FastAPI. Preserve the original ZIP, full reports and notebook.

## Dataset attribution

Official source: https://www.unb.ca/cic/datasets/ids-2017.html

Iman Sharafaldin, Arash Habibi Lashkari, and Ali A. Ghorbani, *Toward Generating a New Intrusion Detection Dataset and Intrusion Traffic Characterization*, ICISSP, 2018.

Read the provider's license and cite this paper in your final report. The official download form showed a server-error message when checked on 14 November 2026; retry or share the error before choosing an unverified mirror.